In [1]:
import cv2
import numpy as np

In [24]:
# Load stereo image pair
img_left = cv2.imread('/data_new/luxiaoxi/dataset/medical_depth/endonerf/pulling_soft_tissues/left/000062.png', cv2.IMREAD_GRAYSCALE)
img_right = cv2.imread('/data_new/luxiaoxi/dataset/medical_depth/endonerf/pulling_soft_tissues/right/000062.png', cv2.IMREAD_GRAYSCALE)

# Initialize ORB detector (or use SIFT/SURF)
orb = cv2.ORB_create()

# Detect keypoints and descriptors
kp1, des1 = orb.detectAndCompute(img_left, None)
kp2, des2 = orb.detectAndCompute(img_right, None)

# Match descriptors using BFMatcher
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

# Sort matches by distance
matches = sorted(matches, key=lambda x: x.distance)

# Extract matched keypoints
pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])

In [25]:
pts1.shape

(172, 2)

In [26]:
pts2.shape

(172, 2)

In [27]:
# Compute Fundamental Matrix
F, mask = cv2.findFundamentalMat(pts1, pts2, cv2.FM_RANSAC, ransacReprojThreshold=1.0, confidence=0.99)

# Filter inlier points
pts1_inliers = pts1[mask.ravel() == 1]
pts2_inliers = pts2[mask.ravel() == 1]

In [28]:
print(F)

[[ 5.05865536e-07 -7.09291281e-05  1.53195980e-02]
 [ 7.09555061e-05  8.77291022e-06 -2.85138237e-03]
 [-1.55854816e-02 -3.65638170e-03  1.00000000e+00]]


In [29]:
# Example intrinsic matrices (replace with your own)
h = 512.0
w = 640.0
f = 569.4682
cx = w / 2
cy = h / 2
K1 = np.array([[f, 0, cx],
               [0, f, cy],
               [0,  0,  1]], dtype=np.float32)
K2 = np.array([[f, 0, cx],
               [0, f, cy],
               [0,  0,  1]], dtype=np.float32)

# Compute Essential Matrix
E = K2.T @ F @ K1

In [30]:
print(E)

[[  0.16404917 -23.0018929   -1.52411432]
 [ 23.01044714   2.84500243  12.58538653]
 [  1.56091513 -13.72864694  -0.12216739]]


In [31]:
# Decompose Essential Matrix
_, R, t, mask = cv2.recoverPose(E, pts1_inliers, pts2_inliers, K1)

# `R` is the rotation matrix (3x3), `t` is the translation vector (3x1)
print("Rotation Matrix:\n", R)
print("Translation Vector:\n", t)

Rotation Matrix:
 [[ 0.99749118  0.05268022 -0.04728778]
 [-0.05370576  0.99834255 -0.02068438]
 [ 0.04611975  0.02317211  0.99866712]]
Translation Vector:
 [[-0.51669063]
 [-0.054278  ]
 [ 0.85444993]]
